In [21]:
import os
import pandas as pd
import qrcode
from qrcode.image.styledpil import StyledPilImage
from qrcode.image.styles.moduledrawers import RoundedModuleDrawer
from qrcode.image.styles.colormasks import SolidFillColorMask
from PIL import Image

def generate_branded_qr(url, output_filename, icon_path=None, fg_color=(0, 0, 0), bg_color=(255, 255, 255), remove_white_bg=False):
    """
    Generates a modern, rounded QR code with an optional embedded icon.
    Includes an auto-transparent background processor for JPG/PNG icons.
    """
    qr = qrcode.QRCode(
        version=5,
        error_correction=qrcode.constants.ERROR_CORRECT_H, 
        box_size=10,
        border=4,
    )
    qr.add_data(url)
    qr.make(fit=True)

    # Apply modern rounded styling
    qr_img = qr.make_image(
        image_factory=StyledPilImage,
        module_drawer=RoundedModuleDrawer(),
        color_mask=SolidFillColorMask(front_color=fg_color, back_color=bg_color)
    ).convert('RGBA')

    # Icon Injection & Transparency Logic
    if icon_path and os.path.exists(icon_path):
        icon = Image.open(icon_path).convert("RGBA")
        
        # --- Auto-Remove White Background ---
        if remove_white_bg:
            datas = icon.getdata()
            new_data = []
            for item in datas:
                # If pixel is close to white (R>240, G>240, B>240), make it transparent
                if item[0] > 240 and item[1] > 240 and item[2] > 240:
                    new_data.append((255, 255, 255, 0)) 
                else:
                    new_data.append(item)
            icon.putdata(new_data)
        
        # Calculate optimal icon size (max 25% of QR width)
        qr_width, qr_height = qr_img.size
        max_icon_size = int(qr_width * 0.25)
        
        icon_width, icon_height = icon.size
        aspect_ratio = icon_width / icon_height
        
        if icon_width > icon_height:
            new_width = max_icon_size
            new_height = int(new_width / aspect_ratio)
        else:
            new_height = max_icon_size
            new_width = int(new_height * aspect_ratio)
            
        icon = icon.resize((new_width, new_height), Image.LANCZOS)
        
        # Center positioning
        pos_x = (qr_width - new_width) // 2
        pos_y = (qr_height - new_height) // 2
        
        # Paste icon
        qr_img.paste(icon, (pos_x, pos_y), icon)

    os.makedirs(os.path.dirname(output_filename), exist_ok=True)
    qr_img.save(output_filename)
    return qr_img

print("✅ QR Engine upgraded with icon_path and Alpha-Channel Processing.")

✅ QR Engine upgraded with icon_path and Alpha-Channel Processing.


In [22]:
import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

def process_manual_list(url_list, output_dir="socmed_qrs", remove_bg=False):
    print(f"Starting lightning-fast batch processing for {len(url_list)} links...")
    os.makedirs(output_dir, exist_ok=True)
    
    def build_qr(item):
        filename = f"{output_dir}/{item['name']}.png"
        icon = item.get('icon_path') 
        generate_branded_qr(item['url'], filename, icon_path=icon, fg_color=(15, 23, 42), remove_white_bg=remove_bg)

    with ThreadPoolExecutor() as executor:
        executor.map(build_qr, url_list)
        
    print(f"✅ Successfully generated {len(url_list)} QR codes in '{output_dir}/'")


def process_excel_batch(excel_path, name_col, url_col, icon_col=None, output_dir="socmed_qrs", remove_bg=False):
    if not os.path.exists(excel_path):
        print(f"❌ Error: Excel file '{excel_path}' not found.")
        return

    df = pd.read_excel(excel_path)
    print(f"Loaded Excel file. Starting parallel processing for {len(df)} records...")
    os.makedirs(output_dir, exist_ok=True)

    def build_qr(row):
        clean_name = str(row[name_col]).replace(" ", "_").replace("/", "-")
        filename = f"{output_dir}/{clean_name}.png"
        
        icon = str(row[icon_col]) if icon_col and pd.notna(row.get(icon_col)) else None
        if icon == 'nan': icon = None
        
        # If the Excel just has the filename (e.g., 'github.jpg'), ensure it points to the icon/ folder
        if icon and not icon.startswith("icon/"):
            icon = f"icon/{icon}"
            
        generate_branded_qr(str(row[url_col]), filename, icon_path=icon, fg_color=(15, 23, 42), remove_white_bg=remove_bg)

    with ThreadPoolExecutor() as executor:
        executor.map(build_qr, [row for _, row in df.iterrows()])

    print(f"✅ Successfully generated {len(df)} QR codes in '{output_dir}/'")

In [ ]:
# --- CONFIGURATION ---
OUTPUT_FOLDER = "socmed_qrs"
REMOVE_WHITE_BG = True  # Set to True to auto-remove white backgrounds from icons (for JPG/PNG) 

# ==========================================
# OPTION A: Run from Manual List (ACTIVE)
# ==========================================
# manual_data = [
#     {"name": "GitHub_Profile", "url": "https://github.com/ifathurrasyid", "icon_path": "icon/github_icon.png"},
#     {"name": "LinkedIn_Profile", "url": "https://linkedin.com/in/ifathurrasyid", "icon_path": "icon/linkedin_icon.png"},
#     {"name": "Instagram_Profile", "url": "https://instagram.com/ifathurrasyid", "icon_path": "icon/instagram_icon.jpg"}
# ]

# process_manual_list(
#     url_list=manual_data, 
#     output_dir=OUTPUT_FOLDER,
#     remove_bg=REMOVE_WHITE_BG
# )

# ==========================================
# OPTION B: Run from Excel File (DISABLED)
# ==========================================
EXCEL_FILE = "socmed_links_database.xlsx"
process_excel_batch(
    excel_path=EXCEL_FILE, 
    name_col="Platform_Name", 
    url_col="Target_URL", 
    icon_col="Icon_File", 
    output_dir=OUTPUT_FOLDER,
    remove_bg=REMOVE_WHITE_BG
)

Loaded Excel file. Starting parallel processing for 15 records...
✅ Successfully generated 15 QR codes in 'socmed_qrs/'
